# Large-Scale PDF Analysis Pipeline

## RAW Extraction of Bookmarks / Table of Contents (TOC)


### Objective

This step constitutes the **deterministic and ultra-fast foundation** of the processing pipeline for large technical PDF documents (which may contain several thousand pages).

Its sole purpose is to extract the internal structure of the bookmarks (*bookmarks / outline / TOC*) exactly as defined in the PDF file, **without any loss or alteration of information**.


### Principles

1. **Pure RAW extraction**: no business interpretation and no heuristics.

2. **Zero deletion**: no entry is removed (even if `page == -1`, the title is empty, there are duplicates, the language is ENG/FRE, the nesting level is deep, etc.).

3. **Zero deduplication / Zero grouping**: the exact order and original position (`index`) are preserved **100%**.

4. **Maximum performance**: uses only `PyMuPDF` (`fitz`) and `doc.get_toc(simple=True)` (execution in a few milliseconds, with no OCR, no LLM, and no visual page analysis).

5. **Handling PDFs without a TOC**: if `get_toc()` returns `[]`, it cleanly produces `sections: []` and `toc_available: false` without inventing any data.


## 1. Configuration

In [1]:
import os
import re
import sys
import json
import pandas as pd
import pymupdf as fitz

from pathlib import Path
from typing import Dict, List, Any, Optional, Union


In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = Path(PROJECT_ROOT / "data" / "raw")

OUTPUT_DIR = Path(PROJECT_ROOT/"results")

PDFS: Dict[str, Path] = {
    "AUSTCOLD": DATA_DIR / "AUSTCOLD.pdf",
    "MYCOM": DATA_DIR / "MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf"
}

# Auto-decouverte des autres PDF dans DATA_DIR s'ils existent
if DATA_DIR.exists():
    for pdf_file in DATA_DIR.glob("*.pdf"):
        key = pdf_file.stem.split()[0]
        if key not in PDFS:
            PDFS[key] = pdf_file

print("Fichiers PDF configures pour extraction :")
for name, path in PDFS.items():
    status = "[TROUVE]" if path.exists() else "[INTROUVABLE]"
    print(f"  * {name:<12} : {path.name} {status}")


Fichiers PDF configures pour extraction :
  * AUSTCOLD     : AUSTCOLD.pdf [TROUVE]
  * MYCOM        : MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf [TROUVE]


## 2. Inspect native metadata

In [3]:
for name, PDF_PATH in PDFS.items():
    doc = fitz.open(PDF_PATH)
    
    print(f"PDF: {name}")
    print("PDF pages:", len(doc))
    print("\nMetadata:")
    for key, value in doc.metadata.items():
        print(f"  {key}: {value}")
    print("-" * 50)
    doc.close()

PDF: AUSTCOLD
PDF pages: 2870

Metadata:
  format: PDF 1.7
  title: 
  author: kennedka
  subject: 
  keywords: 
  creator: Bluebeam PDF Revu
  producer: Bluebeam PDF Library 8.5.1
  creationDate: D:20140304133624+11'00'
  modDate: D:20160412101132+02'00'
  trapped: 
  encryption: None
--------------------------------------------------
PDF: MYCOM
PDF pages: 1068

Metadata:
  format: PDF 1.5
  title: untitled
  author: mara.riga
  subject: 
  keywords: 
  creator: Adobe Acrobat 10.1.6
  producer: macOS Version 14.4.1 (assemblage 23E224) Quartz PDFContext, AppendMode 1.1
  creationDate: D:20140616145557Z00'00'
  modDate: D:20260701105902Z00'00'
  trapped: 
  encryption: None
--------------------------------------------------


## 3. Inspect and extract Bookmarks

In [4]:
for name, PDF_PATH in PDFS.items():
    doc = fitz.open(PDF_PATH)

    print(f"PDF: {name}")
    print('-'*50)
    print()
    
    outline = doc.get_toc(simple=False)
    
    print("Bookmark entries:", len(outline))
    for item in outline[:100]:
        level, title, page = item[:3]
        print(f"L{level:<2} PDF page {page:<4} {title}")
    
    print("-" * 50)
    doc.close()

PDF: AUSTCOLD
--------------------------------------------------

Bookmark entries: 669
L1  PDF page 1    0.0  A5114-OMI-01 Rev 02 Operating and maintenance instructions
L1  PDF page 3    Section 1 Service contact details
L2  PDF page -1   0.00 Operating and maintenance instructions
L2  PDF page 3    0.01  OMI index A5114 - ENG
L2  PDF page 6    0.01  OMI index A5114 - FRE
L2  PDF page 9    1.01  Contact details
L3  PDF page 9    1.01
L3  PDF page 11   1.01  Contact details
L2  PDF page 12   1.02  Standard warning - ENG
L3  PDF page 12   1.02
L3  PDF page 14   1.02  Standard warning
L2  PDF page 15   1.02  Standard warning - FRE
L2  PDF page 16   1.03  Standard safety precautions - ENG
L3  PDF page 16   1.03
L3  PDF page 18   1.03 - Standard safety precautions
L2  PDF page 20   1.03 Standard safety precautions - FRE
L2  PDF page 22   1.04 Standard and support information - ENG
L3  PDF page 22   1.04
L3  PDF page 24   1.04 Standard and support information
L2  PDF page 26   1.04 Standard

The analysis shows that PDF documents may provide different levels of structural information. 

AUSTCOLD already contains a detailed bookmark hierarchy that can be directly reused, while MYCOM has no bookmarks and therefore requires TOC extraction from its content. 

The pipeline will adapt accordingly to preserve the available structure while ensuring that documents without bookmarks can still be organized.

## 4. Extract Bookmarks

In [5]:
from src.document_structure.extract_raw_pdf_toc import extract_raw_toc

raw_toc = extract_raw_toc(PDFS["AUSTCOLD"])

print("Fichier :", raw_toc["filename"])
print("Pages totales :", raw_toc["total_pages"])
print("Entrees de bookmark :", len(raw_toc["sections"]))

bookmark_df = pd.DataFrame(raw_toc["sections"])
display(bookmark_df.head(30))

Fichier : AUSTCOLD.pdf
Pages totales : 2870
Entrees de bookmark : 669


,index,level,title,page
0,0,1,0.0 A5114-OMI-01 Rev 02 Operating and mainten...,1
1,1,1,Section 1 Service contact details,3
2,2,2,0.00 Operating and maintenance instructions,-1
3,3,2,0.01 OMI index A5114 - ENG,3
4,4,2,0.01 OMI index A5114 - FRE,6
5,5,2,1.01 Contact details,9
6,6,3,1.01,9
7,7,3,1.01 Contact details,11
8,8,2,1.02 Standard warning - ENG,12
9,9,3,1.02,12


## 5. TOC detection

### 5.1. Analyze document pages

In [6]:
from src.document_structure.page_analysis import PageAnalyzer


doc = fitz.open(PDFS["MYCOM"])
pages = PageAnalyzer().extract_document(doc)
print(f"Extracted {len(pages)} pages")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Extracted 1068 pages


In [7]:
for p in pages[:10]:
    print(
        f"page {p.page_number}: "
        f"{len(p.raw_text)} chars, "
        f"{len(p.tables)} tables, "
        f"{p.image_count} images, "
        f"scanned={p.is_scanned}"
    )

page 1: 490 chars, 1 tables, 2 images, scanned=False
page 2: 364 chars, 2 tables, 1 images, scanned=False
page 3: 338 chars, 1 tables, 1 images, scanned=False
page 4: 309 chars, 1 tables, 3 images, scanned=False
page 5: 2609 chars, 1 tables, 1 images, scanned=False
page 6: 1251 chars, 1 tables, 1 images, scanned=False
page 7: 310 chars, 1 tables, 3 images, scanned=False
page 8: 0 chars, 0 tables, 1 images, scanned=True
page 9: 367 chars, 1 tables, 1 images, scanned=False
page 10: 321 chars, 1 tables, 1 images, scanned=False


### 5.2. Title detection

In [8]:
from src.document_structure.titles import TitleDetector

title_detector = TitleDetector()

title_candidates_by_page = {
    p.page_number: title_detector.detect(p)
    for p in pages
}

total_candidates = sum(len(c) for c in title_candidates_by_page.values())
print(f"{total_candidates} title candidates across {len(pages)} pages")

3968 title candidates across 1068 pages


In [9]:
for page_number, candidates in list(title_candidates_by_page.items())[:5]:
    for c in candidates:
        print(f"page {page_number} [{c.confidence:.2f}]: {c.text!r} ({c.reasons})")

page 1 [0.65]: 'OPERATING AND MAINTENANCE DATA' (['bold', 'large_font', 'heading_like_text'])
page 1 [0.55]: 'OCP - MAROC PHOSPHORE' (['bold', 'near_top', 'heading_like_text'])
page 1 [0.55]: 'LOCATION:JORF LASFAR - MOROCCOSheet 1' (['bold', 'near_top', 'heading_like_text'])
page 1 [0.55]: 'PLANT: ODI at P1 SITEOCP Doc. Code:455A-REF-PRD- 009' (['bold', 'near_top', 'heading_like_text'])
page 1 [0.55]: 'CLIENT: OCP - MAROC PHOSPHOREVendor Job.N°:2012-123' (['bold', 'near_top', 'heading_like_text'])
page 1 [0.55]: 'N°2 AMMONIA STORAGE TANKSVendor Doc.No.:P1-REF-PRD-12-123-009' (['bold', 'near_top', 'heading_like_text'])
page 1 [0.55]: 'PROJ.N°:Q3600XXXUnit:455A' (['bold', 'near_top', 'heading_like_text'])
page 1 [0.45]: "This document contains Mayekawa's document n°" (['bold', 'heading_like_text'])
page 1 [0.45]: '12OP0003GQB004n° 1066 sheets' (['bold', 'heading_like_text'])
page 2 [0.55]: 'DOCUMENT INDEX' (['bold', 'near_top', 'heading_like_text'])
page 2 [0.55]: 'LOCATION:JORF LASFAR -

Title Detection gives an independent, typography-based signal (bold, font
size, position, heading shape) for what looks like a heading anywhere in
the document, separate from TOC Detection's own keyword/structure-based
scoring.

That's useful here in two ways:

- **Confirming TOC pages themselves** — a page whose top block is a strong
  title candidate matching TOC-like language (e.g. "Table of Contents",
  "Document Index") reinforces `TOCDetector`'s keyword evidence with a
  second, independently-derived signal, rather than relying on keyword
  matching alone.
- **Validating individual entries** — each TOC entry claims a title and a
  target page. Cross-checking that title against the title candidates
  actually detected *on* that target page (Section 4's segmentation step)
  is a way to confirm the entry points somewhere real, rather than trusting
  the TOC text in isolation.

### 5.3. TOC detection

In [10]:
from src.document_structure.toc import TOCDetector


toc_detector = TOCDetector(toc_score_threshold=0.40)  # matches your notebook's threshold

toc_analyses = [toc_detector.analyze_page(p) for p in pages]

flagged = [a for a in toc_analyses if a.is_toc]
print(f"{len(flagged)} pages flagged as TOC")
for a in flagged:
    print(f"  page {a.page_number}: confidence={a.confidence:.2f}, entries={len(a.entries)}")

16 pages flagged as TOC
  page 5: confidence=0.50, entries=47
  page 6: confidence=0.50, entries=22
  page 188: confidence=0.50, entries=36
  page 282: confidence=0.43, entries=4
  page 283: confidence=0.43, entries=4
  page 284: confidence=0.43, entries=6
  page 285: confidence=0.43, entries=6
  page 409: confidence=0.71, entries=77
  page 635: confidence=0.50, entries=3
  page 636: confidence=0.43, entries=18
  page 776: confidence=0.43, entries=18
  page 794: confidence=0.62, entries=31
  page 876: confidence=0.48, entries=0
  page 938: confidence=0.48, entries=0
  page 1000: confidence=0.48, entries=0
  page 1068: confidence=0.50, entries=10


In [11]:
toc_regions = toc_detector.detect_printed_tocs(pages)

print(f"{len(toc_regions)} TOC region(s)")
for region in toc_regions:
    print(f"  pages {region.pages}: {len(region.entries)} entries, confidence={region.confidence:.2f}")

11 TOC region(s)
  pages [5, 6]: 69 entries, confidence=0.50
  pages [188]: 36 entries, confidence=0.50
  pages [282, 283, 284, 285]: 20 entries, confidence=0.43
  pages [409]: 77 entries, confidence=0.71
  pages [635, 636]: 21 entries, confidence=0.50
  pages [776]: 18 entries, confidence=0.43
  pages [794]: 31 entries, confidence=0.62
  pages [876]: 0 entries, confidence=0.48
  pages [938]: 0 entries, confidence=0.48
  pages [1000]: 0 entries, confidence=0.48
  pages [1068]: 10 entries, confidence=0.50


In [12]:
def parse_page_number(reference, reference_kind):
    
    if reference_kind != "page" or not reference:
        return None
    match = re.search(r"\d{1,4}", str(reference))
    return int(match.group(0)) if match else None

toc_entry_results = [
    {
        "section_number": entry.section_number,
        "title": entry.text,
        "level": entry.level,
        "page_number": parse_page_number(entry.printed_page_ref, entry.reference_kind),
    }
    for region in toc_regions
    for entry in region.entries
]

with open(OUTPUT_DIR / "test_toc_entries.json", "w", encoding="utf-8") as f:
    json.dump(toc_entry_results, f, ensure_ascii=False, indent=2)

print(f"saved {len(toc_entry_results)} TOC entries")

saved 282 TOC entries
